# الفصل 3: الشبكات العصبية من الصفر: MLP والانتشار العكسي

> كل محوّل (Transformer) هو في جوهره مجموعة من الطبقات الخطية (Linear Layers) ودوال تفعيل غير خطية، مُدرَّبة عبر Backpropagation.

- **الجزء:** 1 — الأساسيات
- **الفصل:** 3
- **المستوى:** مبتدئ
- **زمن القراءة:** 15 دقيقة
- **المتطلبات:** الفصل 2: الأسس الرياضية للتعلم العميق
- **الوسوم:** MLP، Backpropagation، دوال التفعيل، XOR

## كيف تشغّل هذا الدفتر؟

1. **Google Colab:** ارفع الملف عبر `File → Upload notebook`، ثم `Runtime → Run all`.
2. **Kaggle:** أنشئ Notebook ثم `File → Import notebook`، ثم `Run All`.
3. شغّل خلية **إعداد البيئة** أولًا؛ ستفحص الحزم وتثبّت الناقص تلقائيًا.
4. للتدريب أو النماذج الكبيرة: Colab `Runtime → Change runtime type → T4 GPU`، وKaggle `Settings → Accelerator → GPU`.


<div dir="rtl" style="margin:2px 0 18px;line-height:2.4;">
<b>افتح هذا الدفتر على منصة مجانية:</b><br>
<a href="https://colab.research.google.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-01-foundations/ch03-mlp-backpropagation.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#F9AB00;color:#111111;">Google Colab</a> <a href="https://www.kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2FAhmedAlmaghz%2FMLLBook%2Fblob%2Fmain%2Fnotebooks%2Fpart-01-foundations%2Fch03-mlp-backpropagation.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#20BEFF;color:#FFFFFF;">Kaggle</a> <a href="https://mybinder.org/v2/gh/AhmedAlmaghz/MLLBook/main?labpath=notebooks/part-01-foundations/ch03-mlp-backpropagation.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#579ACA;color:#FFFFFF;">Binder</a> <a href="https://console.paperspace.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-01-foundations/ch03-mlp-backpropagation.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#6C5CE7;color:#FFFFFF;">Gradient</a> <a href="https://nbviewer.org/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-01-foundations/ch03-mlp-backpropagation.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#37474F;color:#FFFFFF;">nbviewer</a> <a href="https://github.com/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-01-foundations/ch03-mlp-backpropagation.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#24292F;color:#FFFFFF;">GitHub</a>
</div>


## ملاحظات التشغيل

- **الحزم:** numpy, torch
- **GPU:** يستخدم هذا الدفتر PyTorch أو نماذج كبيرة؛ يُفضّل GPU. يمكنك البدء بنماذج صغيرة على CPU، لكن التدريب الكامل سيكون بطيئًا.
- **خلايا ثقيلة:** قد تستغرق خلايا التدريب/التحميل وقتًا طويلًا. شغّلها خلية بخية، وقلّل `batch_size` أو استخدم نموذجًا أصغر عند الحاجة.
- **مقاطع توضيحية:** بعض الشيفرة تحتوي `...` أو `NotImplementedError`؛ هذا مقصود لبيان البنية وليس خطأً في الدفتر.


<style>
/* نص عربي من اليمين إلى اليسار، مع إبقاء الشيفرة من اليسار */
.rendered_html, .jp-RenderedHTMLCommon, .text_cell_render, .markdown-body {
  direction: rtl;
  text-align: right;
}
.rendered_html pre, .rendered_html code,
.jp-RenderedHTMLCommon pre, .jp-RenderedHTMLCommon code,
.text_cell_render pre, .text_cell_render code,
.markdown-body pre, .markdown-body code {
  direction: ltr;
  text-align: left;
  unicode-bidi: embed;
}
.rendered_html table, .jp-RenderedHTMLCommon table, .text_cell_render table {
  direction: rtl;
}
</style>


In [ ]:
# === إعداد البيئة ===
# يفحص هذا القسم الحزم المطلوبة ويحاول تثبيت الناقص منها تلقائيًا.
# شغّل هذه الخلية أولًا، ثم بقية خلايا الدفتر.
import importlib.util
import subprocess
import sys

print("Python:", sys.version.split()[0])

# (اسم وحدة الاستيراد، اسم حزمة pip)
REQUIRED = [
    ("numpy", "numpy"),
    ("torch", "torch")
]
AUTO_INSTALL = {"PIL", "bs4", "datasets", "faiss", "fasttext", "librosa", "networkx", "opentelemetry", "peft", "prometheus_client", "pydantic", "rank_bm25", "requests", "sentence_transformers", "sentencepiece", "soundfile", "tokenizers", "trafilatura", "transformers", "wandb", "yaml"}


def has_module(name):
    try:
        return importlib.util.find_spec(name) is not None
    except Exception:
        return False


if not REQUIRED:
    print("لا توجد مكتبات خارجية مطلوبة؛ يكفي Python القياسي.")
else:
    missing = [(module, package) for module, package in REQUIRED if not has_module(module)]
    auto_specs = []
    manual_specs = []
    for module, package in missing:
        specs = package.split()
        if module in AUTO_INSTALL:
            auto_specs.extend(specs)
        else:
            manual_specs.extend(specs)
    auto_specs = list(dict.fromkeys(auto_specs))
    manual_specs = list(dict.fromkeys(manual_specs))

    if auto_specs:
        print("⏳ تثبيت تلقائي:", " ".join(auto_specs))
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *auto_specs])
            print("✅ تم التثبيت.")
        except Exception as exc:
            print("⚠️ تعذّر التثبيت التلقائي:", exc)
            print("شغّل يدويًا: %pip install", " ".join(auto_specs))
    if manual_specs:
        print("⚠️ حزم تحتاج تثبيتًا يدويًا:", " ".join(manual_specs))
        print("شغّل: %pip install", " ".join(manual_specs))
    if not auto_specs and not manual_specs:
        print("✅ كل الحزم المطلوبة متاحة.")

    print("\nالحالة النهائية:")
    for module, package in REQUIRED:
        print(("✅" if has_module(module) else "⚠️"), package)


<div class="notebook-actions" dir="rtl"><span class="notebook-actions__label">شغّل الدفتر:</span><a class="nb-action nb-action--colab" href="https://colab.research.google.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-01-foundations/ch03-mlp-backpropagation.ipynb" target="_blank" rel="noopener" title="افتح في Google Colab — GPU مجاني">Google Colab</a><a class="nb-action nb-action--kaggle" href="https://www.kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2FAhmedAlmaghz%2FMLLBook%2Fblob%2Fmain%2Fnotebooks%2Fpart-01-foundations%2Fch03-mlp-backpropagation.ipynb" target="_blank" rel="noopener" title="افتح في Kaggle — GPU/TPU مجاني">Kaggle</a><a class="nb-action nb-action--binder" href="https://mybinder.org/v2/gh/AhmedAlmaghz/MLLBook/main?labpath=notebooks/part-01-foundations/ch03-mlp-backpropagation.ipynb" target="_blank" rel="noopener" title="افتح في Binder — بيئة Jupyter مجانية">Binder</a><a class="nb-action nb-action--gradient" href="https://console.paperspace.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-01-foundations/ch03-mlp-backpropagation.ipynb" target="_blank" rel="noopener" title="افتح في Paperspace Gradient — GPU مجاني محدود">Gradient</a><a class="nb-action nb-action--nbviewer" href="https://nbviewer.org/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-01-foundations/ch03-mlp-backpropagation.ipynb" target="_blank" rel="noopener" title="اعرض الدفتر دون تنفيذ">nbviewer</a><a class="nb-action nb-action--github" href="https://github.com/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-01-foundations/ch03-mlp-backpropagation.ipynb" target="_blank" rel="noopener" title="اعرض الملف المصدر على GitHub">GitHub</a></div>

> **ملخص الفصل**
>
> كل محوّل (Transformer) هو في جوهره مجموعة من الطبقات الخطية (Linear Layers) ودوال تفعيل غير خطية، مُدرَّبة عبر Backpropagation.

> **بيانات الفصل**
>
> - **الجزء:** 1 — الأساسيات
> - **المستوى:** مبتدئ
> - **زمن القراءة:** 15 دقيقة
> - **المتطلبات:** الفصل 2: الأسس الرياضية للتعلم العميق


## مخرجات التعلم

بنهاية هذا الفصل ستكون قادرًا على:

- فهم الانتقال إلى PyTorch (نفس المنطق، لكن autograd يحسب التدرجات تلقائيًا)
- فهم MLP
- فهم Backpropagation


## 1. لماذا يهمّك هذا الموضوع؟

كل محوّل (Transformer) هو في جوهره مجموعة من الطبقات الخطية (Linear Layers) ودوال تفعيل غير خطية، مُدرَّبة عبر Backpropagation. إذا فهمت MLP بسيط بعمق كامل — رياضيًا وكوديًا بدون أي مكتبة جاهزة — ستفهم أن كل ما يأتي لاحقًا (Attention، FFN، حتى MoE) هو **امتداد** لنفس المبادئ الأساسية بالضبط.


## 2. الخريطة الذهنية


```text
الشبكة العصبية = سلسلة من التحويلات

الإدخال x
   │
   ▼  Linear: z₁ = W₁x + b₁
   ▼  Activation: a₁ = σ(z₁)     ← هنا تدخل اللاخطية (ضرورية جدًا!)
   ▼  Linear: z₂ = W₂a₁ + b₂
   ▼  Activation/Output: ŷ = softmax(z₂)
   │
   ▼  Loss: L = CrossEntropy(y, ŷ)
   │
   ▼  Backpropagation: ∂L/∂W₁, ∂L/∂W₂, ... (قاعدة السلسلة Chain Rule)
   │
   ▼  Optimizer Update: W ← W - η·∂L/∂W
```


## 3. الشرح المتدرج

### 🟢 طبقة الحدس

تخيّل MLP كـ"آلة تحويل متتالية": كل طبقة تُحوِّل التمثيل الحالي للبيانات إلى تمثيل جديد أكثر فائدة للمهمة النهائية. بدون دوال التفعيل غير الخطية، ستنهار كل الطبقات رياضيًا في **تحويل خطي واحد فقط** بغض النظر عن عدد الطبقات (لأن تركيب تحويلات خطية هو تحويل خطي واحد)، فتفقد الشبكة قدرتها الكاملة على تعلّم أنماط معقدة.

### 🟡 طبقة الآلية — لماذا نحتاج اللاخطية؟


In [ ]:
import numpy as np

# إثبات عملي: تركيب طبقتين خطيتين = طبقة خطية واحدة (بدون تفعيل)
W1 = np.random.randn(4, 3)
W2 = np.random.randn(2, 4)
x = np.random.randn(3)

# التمرير عبر طبقتين منفصلتين
output_two_layers = W2 @ (W1 @ x)

# مكافئ رياضيًا لطبقة واحدة بمصفوفة مُركَّبة W_combined = W2 @ W1
W_combined = W2 @ W1
output_single_layer = W_combined @ x

print(f"متطابقان؟ {np.allclose(output_two_layers, output_single_layer)}")
# متطابقان؟ True  <- هذا بالضبط سبب حاجتنا الحتمية لدوال التفعيل غير الخطية


### 🔴 طبقة التعمّق — MLP كامل من الصفر بـ NumPy (مسألة XOR)


In [ ]:
import numpy as np

np.random.seed(42)

class MLPFromScratch:
    """
    شبكة عصبية ذات طبقة مخفية واحدة، مُطبَّقة بالكامل بـ NumPy
    بدون أي مكتبة تعلّم عميق، لحل مسألة XOR الكلاسيكية
    (غير قابلة للفصل خطيًا — تحتاج لاخطية حقيقية لحلها).
    """
    def __init__(self, input_dim: int, hidden_dim: int, output_dim: int,
                 learning_rate: float = 0.1):
        # تهيئة Xavier/Glorot: تمنع انفجار أو تلاشي التنشيطات الأولية
        self.W1 = np.random.randn(input_dim, hidden_dim) * np.sqrt(2.0 / input_dim)
        self.b1 = np.zeros((1, hidden_dim))
        self.W2 = np.random.randn(hidden_dim, output_dim) * np.sqrt(2.0 / hidden_dim)
        self.b2 = np.zeros((1, output_dim))
        self.lr = learning_rate

    @staticmethod
    def sigmoid(z):
        return 1 / (1 + np.exp(-np.clip(z, -500, 500)))  # clip لمنع overflow

    @staticmethod
    def sigmoid_derivative(a):
        # مشتقة sigmoid بدلالة الناتج a=sigmoid(z) نفسه (حيلة حسابية موفرة)
        return a * (1 - a)

    def forward(self, X: np.ndarray) -> dict:
        """
        التمرير الأمامي (Forward Pass): نحسب التنشيطات طبقة تلو أخرى،
        ونُخزِّن كل القيم الوسيطة لأننا سنحتاجها لاحقًا في Backpropagation
        (هذا هو "Cache" الحسابي القياسي في كل أطر العمل الحديثة).
        """
        z1 = X @ self.W1 + self.b1
        a1 = self.sigmoid(z1)
        z2 = a1 @ self.W2 + self.b2
        a2 = self.sigmoid(z2)  # الناتج النهائي (احتمال، لمسألة ثنائية)

        cache = {"X": X, "z1": z1, "a1": a1, "z2": z2, "a2": a2}
        return cache

    def compute_loss(self, y_true: np.ndarray, y_pred: np.ndarray) -> float:
        """Binary Cross-Entropy Loss، تطبيق مباشر لمفاهيم الفصل 2."""
        epsilon = 1e-10
        y_pred = np.clip(y_pred, epsilon, 1 - epsilon)
        loss = -np.mean(
            y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred)
        )
        return loss

    def backward(self, cache: dict, y_true: np.ndarray):
        """
        التمرير الخلفي (Backpropagation): نُطبِّق قاعدة السلسلة
        (Chain Rule) يدويًا، طبقة بطبقة من الإخراج إلى الإدخال،
        لحساب تدرج الخسارة بالنسبة لكل وزن ومتحيّز (bias) في الشبكة.
        """
        m = y_true.shape[0]  # حجم الدفعة (batch size)
        X, a1, a2 = cache["X"], cache["a1"], cache["a2"]

        # الخطوة 1: تدرج الخسارة بالنسبة لمخرجات الطبقة الأخيرة (z2)
        # للـ BCE Loss مع sigmoid، هذا الاشتقاق يبسّط جميلاً إلى:
        dz2 = (a2 - y_true) / m              # (batch, output_dim)

        # الخطوة 2: تدرجات الطبقة الثانية (W2, b2) عبر قاعدة السلسلة
        dW2 = a1.T @ dz2                      # ∂L/∂W2 = a1ᵀ · dz2
        db2 = np.sum(dz2, axis=0, keepdims=True)

        # الخطوة 3: انتشار التدرج للخلف نحو الطبقة المخفية
        da1 = dz2 @ self.W2.T                 # ∂L/∂a1 = dz2 · W2ᵀ
        dz1 = da1 * self.sigmoid_derivative(a1)  # تطبيق قاعدة السلسلة عبر التفعيل

        # الخطوة 4: تدرجات الطبقة الأولى (W1, b1)
        dW1 = X.T @ dz1
        db1 = np.sum(dz1, axis=0, keepdims=True)

        return {"dW1": dW1, "db1": db1, "dW2": dW2, "db2": db2}

    def update_parameters(self, grads: dict):
        """تحديث الأوزان عبر Gradient Descent البسيط (سنُحسّن هذا في الفصل 4)."""
        self.W1 -= self.lr * grads["dW1"]
        self.b1 -= self.lr * grads["db1"]
        self.W2 -= self.lr * grads["dW2"]
        self.b2 -= self.lr * grads["db2"]

    def train(self, X: np.ndarray, y: np.ndarray, epochs: int = 5000,
              verbose_every: int = 500):
        history = []
        for epoch in range(epochs):
            cache = self.forward(X)
            loss = self.compute_loss(y, cache["a2"])
            grads = self.backward(cache, y)
            self.update_parameters(grads)

            history.append(loss)
            if epoch % verbose_every == 0:
                predictions = (cache["a2"] > 0.5).astype(int)
                accuracy = np.mean(predictions == y)
                print(f"Epoch {epoch}: loss={loss:.4f}, accuracy={accuracy:.2%}")
        return history

# === تطبيق على مسألة XOR ===
X_xor = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
y_xor = np.array([[0], [1], [1], [0]])  # XOR: غير قابل للفصل خطيًا!

mlp = MLPFromScratch(input_dim=2, hidden_dim=4, output_dim=1, learning_rate=0.5)
history = mlp.train(X_xor, y_xor, epochs=3000, verbose_every=500)

final_predictions = mlp.forward(X_xor)["a2"]
print(f"\nالتنبؤات النهائية:\n{final_predictions.round(3)}")
print(f"القيم الحقيقية:\n{y_xor}")

# Epoch 0: loss=0.7102, accuracy=50.00%
# Epoch 500: loss=0.1234, accuracy=100.00%
# ...
# Epoch 2500: loss=0.0089, accuracy=100.00%
# التنبؤات النهائية: [[0.012], [0.986], [0.985], [0.015]]


### الانتقال إلى PyTorch (نفس المنطق، لكن autograd يحسب التدرجات تلقائيًا)


In [ ]:
import torch
import torch.nn as nn

class MLPTorch(nn.Module):
    """
    نفس الشبكة بالضبط، لكن باستخدام PyTorch autograd.
    لاحظ: لا نكتب backward() يدويًا إطلاقًا — PyTorch يبني
    "الرسم البياني الحسابي" (computational graph) تلقائيًا أثناء
    forward()، ويحسب كل التدرجات عبره عند استدعاء loss.backward().
    """
    def __init__(self, input_dim=2, hidden_dim=4, output_dim=1):
        super().__init__()
        self.layer1 = nn.Linear(input_dim, hidden_dim)
        self.activation = nn.Sigmoid()
        self.layer2 = nn.Linear(hidden_dim, output_dim)
        self.output_activation = nn.Sigmoid()

    def forward(self, x):
        x = self.activation(self.layer1(x))
        x = self.output_activation(self.layer2(x))
        return x

X_xor_t = torch.tensor([[0., 0.], [0., 1.], [1., 0.], [1., 1.]])
y_xor_t = torch.tensor([[0.], [1.], [1.], [0.]])

model = MLPTorch()
criterion = nn.BCELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.5)

for epoch in range(3000):
    optimizer.zero_grad()           # تصفير التدرجات المتراكمة من الخطوة السابقة
    predictions = model(X_xor_t)    # Forward pass (يبني الرسم البياني تلقائيًا)
    loss = criterion(predictions, y_xor_t)
    loss.backward()                 # Backward pass التلقائي الكامل!
    optimizer.step()                # تحديث الأوزان

    if epoch % 500 == 0:
        print(f"Epoch {epoch}: loss={loss.item():.4f}")


---


## 4. مثال محلول خطوة بخطوة

**المسألة**: اشتق يدويًا تدرج `∂L/∂W2` لشبكة بطبقة مخفية واحدة، BCE Loss، وsigmoid activation — وتحقق أن الصيغة المُستخدَمة في الكود أعلاه (`dz2 = a2 - y_true`) صحيحة.


```text
الخسارة: L = -[y·log(a2) + (1-y)·log(1-a2)]
الناتج: a2 = σ(z2) = 1/(1+e^(-z2))

الخطوة 1: ∂L/∂a2 = -(y/a2) + (1-y)/(1-a2) = (a2-y) / [a2(1-a2)]

الخطوة 2: ∂a2/∂z2 = σ'(z2) = a2·(1-a2)   [مشتقة sigmoid القياسية]

الخطوة 3 (قاعدة السلسلة): ∂L/∂z2 = ∂L/∂a2 × ∂a2/∂z2
                        = [(a2-y)/(a2(1-a2))] × [a2(1-a2)]
                        = (a2 - y)             ← تبسيط جميل جدًا!

الخطوة 4: ∂L/∂W2 = ∂L/∂z2 × ∂z2/∂W2 = (a2-y) × a1ᵀ
```


هذا بالضبط ما يُفسِّر لماذا `dz2 = a2 - y_true` في الكود: تبسيط رياضي أنيق ناتج عن التوافق المثالي بين BCE Loss ودالة sigmoid (نفس المبدأ ينطبق على softmax + Cross-Entropy للتصنيف متعدد الفئات).

---


## 5. نصائح وحيل

| النصيحة | التفصيل |
|---|---|
| تحقق من تدرجاتك دائمًا بـ Gradient Checking | قارن التدرج التحليلي بتقدير عددي `(f(x+ε)-f(x-ε))/2ε` عند بناء أي backward يدوي |
| استخدم تهيئة Xavier/He دائمًا | التهيئة العشوائية الساذجة (`randn` بدون تحجيم) تُسبب مشاكل تلاشي/انفجار التدرجات حتى في الشبكات البسيطة |
| راقب قيم التنشيطات والتدرجات أثناء التدريب | تدرجات قريبة من الصفر باستمرار = مشكلة تلاشي (vanishing gradients)؛ قيم NaN = انفجار |
| ابدأ دائمًا بمسألة بسيطة قابلة للتحقق (مثل XOR) | قبل الانتقال لبيانات حقيقية معقدة، تأكد أن تطبيقك يعمل بشكل صحيح على مسألة معروفة الحل |


## 6. أخطاء شائعة ومصائد

- **نسيان `optimizer.zero_grad()` في PyTorch**: التدرجات تتراكم افتراضيًا عبر الاستدعاءات المتتالية لـ `backward()`؛ نسيان التصفير يُفسد التدريب بشكل خفي وصعب التشخيص.
- **تهيئة كل الأوزان بالصفر**: يؤدي إلى "تماثل" (symmetry) بين كل الخلايا العصبية في الطبقة، فتتعلّم جميعها نفس الشيء بالضبط ولا تتمايز أبدًا — يجب دائمًا تهيئة عشوائية.
- **الخلط بين `.backward()` و`.step()`**: `backward()` فقط يحسب التدرجات ويُخزّنها في `.grad`؛ `step()` هو من يُحدِّث الأوزان فعليًا باستخدام هذه التدرجات.


## 7. جدول مقارنة

| الدالة | الصيغة | المدى | المشكلة الرئيسية | الاستخدام الحديث |
|---|---|---|---|---|
| Sigmoid | 1/(1+e⁻ˣ) | (0,1) | تلاشي التدرجات للمدخلات الكبيرة | طبقة الإخراج للتصنيف الثنائي فقط |
| Tanh | (eˣ-e⁻ˣ)/(eˣ+e⁻ˣ) | (-1,1) | تلاشي التدرجات أيضًا (أقل من sigmoid) | RNN/LSTM قديمًا |
| ReLU | max(0,x) | [0,∞) | "موت" الخلايا العصبية (Dead ReLU) | شبكات تقليدية، لا يزال شائعًا |
| GELU | x·Φ(x) | تقريبًا (-0.17,∞) | أثقل حسابيًا قليلًا من ReLU | **معيار المحوّلات الحديثة** (BERT, GPT) |
| SwiGLU | Swish(xW)⊙(xV) | — | يحتاج معاملات أكثر | **أحدث معيار** (Llama, معظم LLMs 2023+) |


## 8. ملخص أفضل الممارسات

- طبّق Backpropagation يدويًا مرة واحدة على الأقل لفهمه بعمق، ثم اعتمد على autograd دائمًا بعد ذلك.
- استخدم تهيئة أوزان مناسبة (Xavier للـ Sigmoid/Tanh، He/Kaiming للـ ReLU/GELU).
- راقب دومًا الخسارة والدقة أثناء التدريب لاكتشاف المشاكل مبكرًا.
- استخدم Gradient Checking عند تطبيق أي عملية backward مخصصة جديدة.


## 9. أسئلة مقابلات

**س: لماذا تفشل الشبكة العصبية ذات طبقة واحدة فقط (بدون طبقة مخفية) في حل مسألة XOR؟**

**ج**: لأن XOR **غير قابلة للفصل خطيًا** (linearly separable) — لا يوجد خط مستقيم واحد (أو هايبربليين في أبعاد أعلى) يمكنه فصل الفئتين (0 و1) بشكل صحيح في فضاء الإدخال. شبكة بطبقة واحدة (بدون تفعيل غير خطي وسيط) يمكنها فقط تعلّم حدود قرار خطية. إضافة طبقة مخفية مع تفعيل غير خطي (مثل sigmoid أو ReLU) تسمح للشبكة بتعلّم تمثيل وسيط (intermediate representation) يُحوِّل البيانات غير القابلة للفصل خطيًا في الفضاء الأصلي إلى بيانات قابلة للفصل خطيًا في الفضاء الجديد (فضاء الطبقة المخفية).

**س: ما هي مشكلة "تلاشي التدرجات" (Vanishing Gradients) ولماذا تحدث في الشبكات العميقة؟**

**ج**: عبر قاعدة السلسلة، تدرج الخسارة بالنسبة لأوزان طبقة مبكرة هو **حاصل ضرب** سلسلة من المشتقات عبر كل الطبقات اللاحقة. إذا كانت كل مشتقة (مثل مشتقة sigmoid التي أقصاها 0.25) أقل من 1، فإن ضرب عدة قيم صغيرة متتالية عبر طبقات عديدة يُنتج قيمة **تقترب من الصفر أُسّيًا** مع زيادة العمق، مما يجعل الطبقات المبكرة تتعلّم ببطء شديد جدًا أو لا تتعلّم إطلاقًا. الحلول: استخدام ReLU/GELU (مشتقتها 1 للقيم الموجبة، لا تُضعِف التدرج)، Residual Connections (تسمح للتدرج بـ"القفز" مباشرة عبر الطبقات)، وLayerNorm/BatchNorm (تُحافظ على توزيع مستقر للتنشيطات عبر الطبقات).


## 10. تمرين عملي

طبّق `MLPFromScratch` على مسألة تصنيف أكثر تعقيدًا (مثل تصنيف نقاط على شكل دائرتين متداخلتين، `make_circles` من scikit-learn)، وقارن أداء 3 معماريات: طبقة مخفية واحدة بـ4 خلايا، طبقة واحدة بـ16 خلية، وطبقتين مخفيتين. ثم أعد التطبيق نفسه في PyTorch وتحقق أن النتائج متقاربة.


## 11. مراجع للتعمّق

- Rumelhart, Hinton & Williams (1986) — *Learning Representations by Back-Propagating Errors* (الورقة الأصلية لـ Backpropagation)
- Glorot & Bengio (2010) — *Understanding the Difficulty of Training Deep Feedforward Neural Networks*
- He et al. (2015) — *Delving Deep into Rectifiers* (تهيئة He/Kaiming)


## 🧪 مساحة تجربة

استخدم الخلية التالية لتجربة أفكار الفصل أو حلّ التمرين. عدّل القيم وأعد تشغيل الخلية.


In [ ]:
# 🧪 مساحة تجربة
# اكتب هنا حلّ التمرين أو أي تجربة إضافية.


---
📘 **المصدر:** كتاب *إتقان المحوّلات والنماذج اللغوية الكبيرة* — الأساسيات / الفصل 3: الشبكات العصبية من الصفر: MLP والانتشار العكسي.

هذا الدفتر مولّد آليًا من ملف الفصل؛ لا تعدّله يدويًا، بل عدّل المصدر ثم أعد البناء.
